In [1]:
!pip install snowflake-connector-python[pandas] --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 5.7 MB/s eta 0:00:00


In [2]:
import snowflake.connector
import pandas as pd
import matplotlib.pyplot as plt
SNOWFLAKE_ACCOUNT  = 'OKHKGRZ-MA58454'  # Example: 'ab12345.ap-south-1.aws'
SNOWFLAKE_USER     = 'ALLIMUTHU19'             # The username or email you registered with
SNOWFLAKE_PASSWORD = 'Allimuthu@2006'             # Your Snowflake password
SNOWFLAKE_ROLE     = 'ACCOUNTADMIN'              # Role that gives full access
SNOWFLAKE_WH       = 'COMPUTE_WH'

conn = snowflake.connector.connect(
    account     = SNOWFLAKE_ACCOUNT,
    user        = SNOWFLAKE_USER,
    password    = SNOWFLAKE_PASSWORD,
    role        = SNOWFLAKE_ROLE,
    warehouse   = SNOWFLAKE_WH
)

cursor = conn.cursor()

print("Connection to Snowflake was successful")

Connection to Snowflake was successful


In [3]:
cursor.execute("CREATE DATABASE IF NOT EXISTS INTERNSHIP_DATABASE")
cursor.execute("USE DATABASE INTERNSHIP_DATABASE")

print("Database INTERNSHIP_DB created and selected.")

Database INTERNSHIP_DB created and selected.


In [4]:
cursor.execute("CREATE SCHEMA IF NOT EXISTS ANALYTICS")
cursor.execute("USE SCHEMA ANALYTICS")
print("Schema analytics created and selected")


Schema analytics created and selected


In [5]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS STUDENT_PERFORMANCE(
    student_id INT,
    student_name VARCHAR(100),
    subject_name VARCHAR(50),
    score FLOAT,
    grade VARCHAR(5),
    attendance_pct FLOAT,
    exam_date DATE
)
"""

cursor.execute(create_table_sql)
print("Table STUDENT_PERFORMANCE create successfully")

Table STUDENT_PERFORMANCE create successfully


In [7]:
print("Tables in analytics schema:")
cursor.execute("SHOW TABLES")
for row in cursor.fetchall():
  print(" ",row[1])
print()

print("columns in STUDENT_PERFORMANCE")
cursor.execute("DESCRIBE TABLE STUDENT_PERFORMANCE")
for row in cursor.fetchall():
  print(f" Column: {row[0]:25s}Type:{row[1]}")

Tables in analytics schema:
  STUDENT_PERFORMANCE

columns in STUDENT_PERFORMANCE
 Column: STUDENT_ID               Type:NUMBER(38,0)
 Column: STUDENT_NAME             Type:VARCHAR(100)
 Column: SUBJECT_NAME             Type:VARCHAR(50)
 Column: SCORE                    Type:FLOAT
 Column: GRADE                    Type:VARCHAR(5)
 Column: ATTENDANCE_PCT           Type:FLOAT
 Column: EXAM_DATE                Type:DATE


In [9]:
data = {
    'student_id': [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10
    ],
    'student_name': [
        'Aarav Sharma', 'Priya Nair', 'Rohan Mehta', 'Sneha Iyer', 'Arjun Reddy',
        'Kavya Patel', 'Vikram Singh', 'Ananya Das', 'Kiran Joshi', 'Meera Rao',
        'Aarav Sharma', 'Priya Nair', 'Rohan Mehta', 'Sneha Iyer', 'Arjun Reddy',
        'Kavya Patel', 'Vikram Singh', 'Ananya Das', 'Kiran Joshi', 'Meera Rao',
        'Aarav Sharma', 'Priya Nair', 'Rohan Mehta', 'Sneha Iyer', 'Arjun Reddy',
        'Kavya Patel', 'Vikram Singh', 'Ananya Das', 'Kiran Joshi', 'Meera Rao'
    ],
    'subject': [
        'Mathematics', 'Mathematics', 'Mathematics', 'Mathematics', 'Mathematics',
        'Mathematics', 'Mathematics', 'Mathematics', 'Mathematics', 'Mathematics',
        'Science', 'Science', 'Science', 'Science', 'Science',
        'Science', 'Science', 'Science', 'Science', 'Science',
        'English', 'English', 'English', 'English', 'English',
        'English', 'English', 'English', 'English', 'English'
    ],
    'score': [
        88.5, 92.0, 74.0, 85.5, 67.0, 95.0, 78.5, 81.0, 70.5, 90.0,
        76.0, 88.0, 69.5, 91.0, 72.0, 84.5, 65.0, 77.0, 83.5, 94.0,
        82.0, 79.5, 88.0, 73.0, 91.5, 68.5, 86.0, 93.0, 75.0, 80.5
    ],
    'grade': [
        'A', 'A+', 'B', 'A', 'C+', 'A+', 'B+', 'A', 'B', 'A+',
        'B+', 'A', 'C+', 'A+', 'B', 'A', 'C', 'B+', 'A', 'A+',
        'A', 'B+', 'A', 'B', 'A+', 'C+', 'A', 'A+', 'B+', 'A'
    ],
    'attendance_pct': [
        95.0, 98.0, 82.0, 90.0, 75.0, 99.0, 85.0, 88.0, 79.0, 96.0,
        92.0, 97.0, 80.0, 94.0, 77.0, 91.0, 72.0, 86.0, 88.0, 98.0,
        89.0, 83.0, 95.0, 78.0, 93.0, 74.0, 91.0, 99.0, 81.0, 87.0
    ],
    'exam_date': [
        '2024-03-15', '2024-03-15', '2024-03-15', '2024-03-15', '2024-03-15',
        '2024-03-15', '2024-03-15', '2024-03-15', '2024-03-15', '2024-03-15',
        '2024-03-16', '2024-03-16', '2024-03-16', '2024-03-16', '2024-03-16',
        '2024-03-16', '2024-03-16', '2024-03-16', '2024-03-16', '2024-03-16',
        '2024-03-17', '2024-03-17', '2024-03-17', '2024-03-17', '2024-03-17',
        '2024-03-17', '2024-03-17', '2024-03-17', '2024-03-17', '2024-03-17'
    ]
}
df = pd.DataFrame(data)
df.to_csv('student_performance')
df.head()

,student_id,student_name,subject,score,grade,attendance_pct,exam_date
0,1,Aarav Sharma,Mathematics,88.5,A,95.0,2024-03-15
1,2,Priya Nair,Mathematics,92.0,A+,98.0,2024-03-15
2,3,Rohan Mehta,Mathematics,74.0,B,82.0,2024-03-15
3,4,Sneha Iyer,Mathematics,85.5,A,90.0,2024-03-15
4,5,Arjun Reddy,Mathematics,67.0,C+,75.0,2024-03-15


In [11]:
insert_sql = """
INSERT INTO STUDENT_PERFORMANCE (student_id, student_name, subject_name, score, grade, attendance_pct, exam_date)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

rows = {tuple(row) for row in df.values.tolist()}
cursor.executemany(insert_sql, rows)

print(f"{len(rows)} rows inserted into STUDENT_PERFORMANCE....")

30 rows inserted into STUDENT_PERFORMANCE....


In [12]:
def get_schema(conn, table_name="STUDENT_PERFORMANCE"):
  cursor = conn.cursor()
  cursor.execute(f"DESCRIBE TABLE {table_name}")
  columns = cursor.fetchall()

  schema_lines = [f"Table : {table_name}"]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f" -{col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()
  schema_lines.append("\nSample rows (first 3):")

  for row in sample_rows:
    schema_lines.append(f" {row}")

  return "\n".join(schema_lines)

schema = get_schema(conn)
print(schema)

Table : STUDENT_PERFORMANCE
Columns:
 -NUMBER(38,0) (COLUMN)
 -VARCHAR(100) (COLUMN)
 -VARCHAR(50) (COLUMN)
 -FLOAT (COLUMN)
 -VARCHAR(5) (COLUMN)
 -FLOAT (COLUMN)
 -DATE (COLUMN)

Sample rows (first 3):
 (5, 'Arjun Reddy', 'Mathematics', 67.0, 'C+', 75.0, datetime.date(2024, 3, 15))
 (4, 'Sneha Iyer', 'English', 73.0, 'B', 78.0, datetime.date(2024, 3, 17))
 (8, 'Ananya Das', 'English', 93.0, 'A+', 99.0, datetime.date(2024, 3, 17))


In [13]:
!pip install groq --q
import os

from groq import Groq
import re
print("Libraries installled successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.3 MB/s eta 0:00:00
Libraries installled successfully


In [14]:
os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

model = "llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using model: {model}")

Groq client initialized successfully
Using model: llama-3.1-8b-instant
